In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt

from shapely.geometry import box

from skimage.registration import phase_cross_correlation

In [2]:
from atlas.stitching import get_tiles_dataframe, stitch_ATLAS_tiles
from atlas.stitching import get_total_canvas_size, get_overlap_relative, mask_low_and_saturation, first_last_true, extract_s_number

In [3]:
series_folder = Path(r".\data\test-series")
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\atlas_L32-10-1_data\session_462189531\ROI-w1-20nm-bsd")
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\atlas_L32-10-1_data\session_2053739366\ROI-w1-20nm-bsd")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_L32-10-2\atlas_L32-10-2_20250410_data\session_955928929\w3-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_L32-10-2\atlas_L32-10-2_20250410_data\session_1395343682\w2-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_862126381\Section Set 3")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_402716765\ta31-roi1-2")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA29\Site 3")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA30\ROI")
series_folder = Path(r'E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_olive_2\zeinab-olive-2-main_data\session_598049588\roi-01')
series_folder = Path(r'E:\PROJECTS\EM\Filipa\M2-2\whole sample sections 20251112_data\session_1077734160\Site 2')
series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\test\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\Filipa-M2-2-20251127\sections 8-12_data\session_1422202770\Site 1")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant-3-wafer-1\alma-plant-3_data\session_422002753\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant2-wafer-1\plant 2_data\session_1115605826\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant4-wafer-1\alma-plant-4_data\session_1084385689\Site 1")
series_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        tif_files = list(folder.glob("*.tif"))
        if tif_files:  # Check if the list is not empty
            print(f"Found series folder: {folder.name} (contains {len(tif_files)} .tif files)")
            series_list.append(folder)

Found series folder: S_001_2087154489 (contains 2 .tif files)
Found series folder: S_002_860056100 (contains 2 .tif files)
Found series folder: S_003_1010050745 (contains 2 .tif files)
Found series folder: S_004_315123615 (contains 2 .tif files)
Found series folder: S_005_400247288 (contains 2 .tif files)
Found series folder: S_006_489722842 (contains 2 .tif files)
Found series folder: S_007_352833351 (contains 2 .tif files)
Found series folder: S_008_641985125 (contains 2 .tif files)
Found series folder: S_009_1197485053 (contains 2 .tif files)
Found series folder: S_010_1334056440 (contains 2 .tif files)
Found series folder: S_011_548458973 (contains 2 .tif files)
Found series folder: S_012_1251364691 (contains 2 .tif files)
Found series folder: S_013_276225188 (contains 2 .tif files)
Found series folder: S_014_624632118 (contains 2 .tif files)
Found series folder: S_015_2030043666 (contains 2 .tif files)
Found series folder: S_016_179181665 (contains 2 .tif files)
Found series folde

In [4]:
from collections import defaultdict, deque

def add_tile_overlap_columns(df, geometry_column="geometry"):
    """
    Computes pairwise overlap between tiles based on spatial geometry and adds two new columns:
    - 'overlaps_bool': List of booleans per row indicating which other tiles it overlaps with.
    - 'overlap_percent': List of integers per row showing the percentage overlap with each tile.

    Notes:
    - A tile is not considered to overlap with itself (overlap is False and 0%).
    - The percentage is computed relative to the tile's own area (not the intersected tile's).

    Parameters:
        df (pd.DataFrame): A DataFrame containing a column with shapely geometries.
        geometry_column (str): Name of the column containing shapely geometry boxes. Default is "geometry".

    Returns:
        pd.DataFrame: The same DataFrame with two new columns added.
    """

    # --- Validations ---
    assert geometry_column in df.columns, f"'{geometry_column}' column not found in DataFrame."
    from shapely.geometry.base import BaseGeometry
    assert all(isinstance(g, BaseGeometry) for g in df[geometry_column]), \
        f"All entries in '{geometry_column}' must be shapely geometry objects."

    n = len(df)
    overlaps_bool = []
    overlaps_percent = []

    for current_idx in range(n):
        current_box = df[geometry_column][current_idx]
        bool_row = []
        percent_row = []
        for query_idx in range(n):
            if current_idx == query_idx:
                bool_row.append(False)
                percent_row.append(0)
                continue

            query_box = df[geometry_column][query_idx]
            if current_box.intersects(query_box):
                overlap_area = current_box.intersection(query_box).area
                overlap_percent = int(round((overlap_area / current_box.area) * 100))
                bool_row.append(True)
                percent_row.append(overlap_percent)
            else:
                bool_row.append(False)
                percent_row.append(0)

        overlaps_bool.append(bool_row)
        overlaps_percent.append(percent_row)

    df["overlaps_bool"] = overlaps_bool
    df["overlap_percent"] = overlaps_percent

    return df

def match_tiles(input_df, reference_idx, min_overlap_percent=2):
    """
    Compute stitching cost and pixel-shift vectors between a reference tile and all other tiles.

    Parameters
    ----------
    input_df : pandas.DataFrame
        DataFrame containing tile metadata. Must include the following columns:
        - 'geometry' : shapely.geometry.Polygon bounding box of the tile
        - 'ImageWidth' : width in pixels
        - 'ImageHeight' : height in pixels
        - 'Filename' : TIFF filename
        - 'raw_data_folder' : pathlib.Path to folder containing the raw TIFF
        - 'overlap_percent' : list-like of overlap percentages with all other tiles
    reference_idx : int
        Index of the tile used as the reference for cost and shift computation.
    min_overlap_percent : float, optional (default=2)
        Minimum overlap (%) required to attempt matching.

    Returns
    -------
    cost_list : list of float
        A list of stitching costs, one per tile.
    shift_list : list of np.ndarray, shape (2,)
        A list of detected pixel shifts (row, col) from each tile to the reference tile.
        If no overlap or too small overlap, shift is np.zeros(2).
    """

    # ------------------------------------------------------------------
    # Assertions: Validate DataFrame structure
    # ------------------------------------------------------------------
    required_cols = [
        'geometry', 'ImageWidth', 'ImageHeight',
        'Filename', 'raw_data_folder', 'overlap_percent'
    ]
    for col in required_cols:
        assert col in input_df.columns, f"Missing required column: '{col}'"

    assert 0 <= reference_idx < len(input_df), "reference_idx is out of DataFrame bounds"
    assert isinstance(min_overlap_percent, (int, float)), "min_overlap_percent must be numeric"

    # ------------------------------------------------------------------
    # Setup reference tile
    # ------------------------------------------------------------------
    row_ref = input_df.iloc[reference_idx]
    geometry_ref = row_ref['geometry']
    w_ref = row_ref['ImageWidth']
    h_ref = row_ref['ImageHeight']

    # Load dtype from TIFF
    ref_tif_path = row_ref.raw_data_folder.joinpath(Path(row_ref.Filename).name)
    with tiff.TiffFile(ref_tif_path) as tif:
        image_dtype = tif.pages[0].dtype

    print(f"\nProcessing reference tile {reference_idx}...")

    # Prepare outputs
    n_tiles = len(input_df)
    cost_list = []
    shift_list = []

    # ------------------------------------------------------------------
    # Compare reference tile to all other tiles
    # ------------------------------------------------------------------
    for query_idx in range(n_tiles):
        print(f"\nComparing reference {reference_idx} to tile {query_idx}...")

        row_mov = input_df.iloc[query_idx]
        overlap_percentage = row_ref.overlap_percent[query_idx]

        print(f"Overlap %: {overlap_percentage}")

        # Not enough overlap → default cost/shift
        if overlap_percentage < min_overlap_percent:
            print("Too little overlap: assigning cost=1.0 and shift=[0,0]")
            cost_list.append(np.float64(1.0))
            shift_list.append(np.zeros(2))
            continue

        # ------------------------------------------------------------------
        # Compute overlapping bounding boxes
        # ------------------------------------------------------------------
        geometry_mov = row_mov['geometry']
        mov_tif_path = row_ref.raw_data_folder.joinpath(Path(row_mov.Filename).name)

        ref_box, mov_box = get_overlap_relative(
            box_reference=geometry_ref,
            box_moving=geometry_mov
        )

        # ------------------------------------------------------------------
        # Load overlapping region from reference image
        # ------------------------------------------------------------------
        with tiff.TiffFile(ref_tif_path) as tif:
            y0_tmp, y1_tmp = int(ref_box.bounds[1]), int(ref_box.bounds[3])
            y0, y1 = h_ref - y1_tmp, h_ref - y0_tmp  # flip correction
            x0, x1 = int(ref_box.bounds[0]), int(ref_box.bounds[2])
            crop_ref = np.flipud(tif.asarray()[y0:y1, x0:x1])

        # ------------------------------------------------------------------
        # Load overlapping region from moving image
        # ------------------------------------------------------------------
        with tiff.TiffFile(mov_tif_path) as tif:
            y0_tmp, y1_tmp = int(mov_box.bounds[1]), int(mov_box.bounds[3])
            y0, y1 = h_ref - y1_tmp, h_ref - y0_tmp
            x0, x1 = int(mov_box.bounds[0]), int(mov_box.bounds[2])
            crop_mov = np.flipud(tif.asarray()[y0:y1, x0:x1])

        # ------------------------------------------------------------------
        # Mask & threshold computation
        # ------------------------------------------------------------------
        mask_ref = ~mask_low_and_saturation(crop_ref)
        mask_mov = ~mask_low_and_saturation(crop_mov)

        current_vals = np.concatenate([
            crop_ref[mask_ref].ravel(),
            crop_mov[mask_mov].ravel()
        ])

        pix_mean = current_vals.mean()
        pix_std = current_vals.std()
        pix_th = pix_mean + 2 * pix_std

        mask_mov = np.logical_and(mask_mov, crop_mov > pix_th)
        mask_ref = np.logical_and(mask_ref, crop_ref > pix_th)

        mask_pixels = mask_mov.sum()
        mask_pixels_per = mask_pixels / mask_mov.size

        # ------------------------------------------------------------------
        # Phase cross-correlation (shift detection)
        # ------------------------------------------------------------------
        detected_shift, _, _ = phase_cross_correlation(
            crop_ref,
            crop_mov,
            reference_mask=mask_ref,
            moving_mask=mask_mov
        )

        print(f"Detected pixel offset (row, col): {-detected_shift}")

        # ------------------------------------------------------------------
        # Cost computation
        # ------------------------------------------------------------------
        ref_std = crop_ref[mask_ref].std()
        mov_std = crop_mov[mask_mov].std()
        avg_std = (ref_std + mov_std) / 2

        cost = 1.0 / (1e-6 + avg_std * mask_pixels_per * overlap_percentage)
        cost_list.append(cost)
        shift_list.append(detected_shift)

    return cost_list, shift_list

def find_all_paths_to_root(mst_sparse, root):
    mst_edges = np.transpose(mst_sparse.nonzero())
    adj = defaultdict(list)

    for i, j in mst_edges:
        adj[i].append(j)
        adj[j].append(i)  # because it's an undirected tree
    #adj = build_adjacency_from_mst(mst)

    paths = {}

    def dfs(node, parent, path):
        path = path + [node]
        paths[node] = path[::-1]  # reversed: from node to root
        for neighbor in adj[node]:
            if neighbor != parent:
                dfs(neighbor, node, path)

    dfs(root, None, [])
    return paths

def path_to_pairs(path):
    """
    Given a list of tile indices representing a stitching path,
    return a list of (moving_tile, reference_tile) pairs.
    The last pair is a self-reference (e.g., (0, 0)).
    """
    pairs = []
    for i in range(len(path)):
        current = path[i]
        if i < len(path) - 1:
            next_ = path[i + 1]
        else:
            next_ = current  # last: self-reference
        pairs.append((current, next_))
    return pairs

def build_adjacency_matrix_from_costs(df, cost_column='stitching_costs'):
    """
    Construct an adjacency matrix from stitching costs to be used for MST computation.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing stitching cost vectors for each tile.
    cost_column : str, optional
        Name of the column containing lists/arrays of stitching costs (default: 'stitching_costs').

    Returns
    -------
    adj_matrix : np.ndarray, shape (n_tiles, n_tiles)
        Adjacency matrix with np.inf for non-edges and stitching costs for valid connections.
    """
    assert cost_column in df.columns, f"Column '{cost_column}' not found in DataFrame."
    
    n_tiles = len(df)
    adj_matrix = np.ones((n_tiles, n_tiles))  # start with 1s (worst or no connection)

    # Fill the matrix with actual stitching costs
    for i in range(n_tiles):
        cost_vector = df['stitching_costs'][i]
        for j in range(n_tiles):
            if i != j and cost_vector[j] < 1.0:
                adj_matrix[i][j] = cost_vector[j]
            else:
                adj_matrix[i][j] = np.inf  # No edge (self or non-overlap)
    
    return adj_matrix

def build_transform_dict_from_mst(mif_tile_df, mst, reference_tile=0, stitching_shift_column='stitching_shifts'):
    """
    Build a dictionary of stitching transformations for all tiles, using a minimum spanning tree (MST)
    and a selected reference tile.

    Parameters
    ----------
    mif_tile_df : pd.DataFrame
        DataFrame with per-tile data, must contain a column of stitching shifts (vectors),
        and be indexable by tile index.
    mst : scipy.sparse.csr_matrix
        Minimum spanning tree in sparse matrix form (output of minimum_spanning_tree).
    reference_tile : int, optional
        Tile index to use as the reference (default is 0).
    stitching_shift_column : str, optional
        Column name in the DataFrame that stores per-tile stitching shift vectors (default: 'stitching_shifts').

    Returns
    -------
    transform_dict : dict
        Dictionary with keys like "t21" meaning transform from tile 2 to tile 1, and values as np.array of shape (2,).
    """

    # The MST will be used to calculate the transofrmation matrices between each tile and a reference tile.
    # For the moment I just pick 0 as reference but maybe there is a better way, in general I dont think it matters much.

    # User input reference_tile and mst, output transform dictionary
    paths_to_reference = find_all_paths_to_root(mst, reference_tile)

    mst_dense = mst.toarray()
    # print("MST adjacency matrix (only connected edges):")
    # print(mst_dense)

    # now calculate inital transforms and store them in a dictionary, by initial I mean they are given by the local conection between tiles
    transform_dict = {}
    mst_edges = np.transpose(mst.nonzero())  # array of (idx_ref, idx_mov) pairs

    for idx_ref, idx_mov in mst_edges:

        cost = mst_dense[idx_ref][idx_mov]
        # print(f"Edge: Reference Tile {idx_ref} - Moving Tile {idx_mov} with cost {cost}")

        # fetch the transform from the dataframe, in this case is a simple translation so a x,y shift vector
        row_ref = mif_tile_df.iloc[idx_ref]
        t_value = row_ref[stitching_shift_column][idx_mov]

        # Create key in the transform dictionary using the format "t{mov}{ref}"
        key = f"t{idx_mov}{idx_ref}"
        transform_dict[key] = t_value

        # now add the oposite for symetry, so we can move back and forth
        key = f"t{idx_ref}{idx_mov}"
        transform_dict[key] = -t_value

    # now add the special case of the reference tile to itsef as all paths should lead to this
    key = f"t{reference_tile}{reference_tile}"
    transform_dict[key] = np.zeros(2)

    # now add all the long range connections where I need to traverse more than one tile edge
    n_tiles = mif_tile_df.shape[0]
    for query_tile in range(n_tiles):
        t_desired = f"t{query_tile}{reference_tile}"
        if t_desired in transform_dict:
            pass
            # print(f"Key {t_desired} exists!, I can do the transform!")
        else:
            # print(f"Key {t_desired} does not exists!, calculating path")
            chain = paths_to_reference[query_tile]

            t_value = np.zeros(2)
            pairs = path_to_pairs(chain)
            for pair in pairs:
                key = f"t{pair[0]}{pair[1]}"
                # print(transform_dict[key])
                t_value = t_value + transform_dict[key]

            transform_dict[t_desired] = t_value

    return transform_dict

def apply_transforms_and_stitch(mif_tile_df, transform_dict, reference_tile=0):
    """
    Apply computed transforms to all tiles and create a single stitched image.

    Parameters
    ----------
    mif_tile_df : pd.DataFrame
        DataFrame containing tile metadata, including:
        - 'Filename' : filename of the tile
        - 'raw_data_folder' : path to folder containing TIFF files
        - 'geometry' : shapely box describing where tile is placed
    transform_dict : dict
        Dictionary of shifts (2D vectors) keyed as 'tXY' meaning from tile X to tile Y.
    reference_tile : int,
        Index of the reference tile (default is 0).

    Returns
    -------
    stitched_img : np.ndarray
        The final stitched image with all tiles placed using their transforms.
    """

    # apply transform, user inputs are transform_dict and mif_tile_df, output is the stitched_img

    # TODO: improve canvas size estimation using max shift, but for now use fixed version
    max_shift_pixels = max(np.abs(val).max() for val in transform_dict.values())
    # print(max_shift_pixels)

    total_img_width, total_img_height = get_total_canvas_size(mif_tile_df)

    stitched_img = None  # initialize later

    for index, row in mif_tile_df.iterrows():
        print(f"\nIndex: {index}, Filename: {row['Filename']}")

        t_key = f"t{index}{reference_tile}"
        print(t_key)

        t_val = transform_dict[t_key]
        print(t_val.shape)

        # Load current tile image and flip vertically
        tif_current = row.raw_data_folder.joinpath(Path(row.Filename).name)
        img_current = np.flipud(tiff.imread(tif_current))  # Flip image

        # Initialize stitched canvas on first iteration
        if index == 0:
            stitched_img = np.zeros(
                [total_img_height, total_img_width],
                dtype=img_current.dtype
            )
            #extracted_number = extract_s_number(tif_current)

        # Compute shifted bounding box position
        box_mov = row.geometry
        y0 = int(box_mov.bounds[1] + t_val[0])
        y1 = int(box_mov.bounds[3] + t_val[0])
        x0 = int(box_mov.bounds[0] + t_val[1])
        x1 = int(box_mov.bounds[2] + t_val[1])

        stitched_img[y0:y1, x0:x1] = img_current

    return stitched_img


In [5]:
from atlas.io import extract_s_number
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from collections import defaultdict, deque
import json

buffer_in_microns = 5
#TODO: Implement the max shift check
max_shift_in_pixles = 400

for raw_data_folder in series_list:
    # Check if files in the folder have the ".ve-tie" extension
    tie_file = None
    mif_file = None
    for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
        #if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
        #    print(f"File with '.ve-tie' extension found: {file.name}")
        #    tie_file = file
        if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file
            mif_tile_df = get_tiles_dataframe(mif_file, buffer_microns=buffer_in_microns)
            
            first_tif_path = raw_data_folder.joinpath(Path(mif_tile_df.iloc[0]['Filename']).name)
            extracted_number = extract_s_number(first_tif_path)
            # Define the output file path
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            output_cc_path = raw_data_folder.parent.joinpath(f"phaseCC_stitching_S_{extracted_number}.csv")
            output_jason_path = raw_data_folder.parent.joinpath(f"transforms_S_{extracted_number}.json")

            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")
                #stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=max_shift_in_pixles)

                try:
                    mif_tile_df = add_tile_overlap_columns(mif_tile_df)
                    # add info to the DF so we know where to find the images after they have been moved out of the scope
                    mif_tile_df['raw_data_folder'] = raw_data_folder
                    
                    # Calculate the costs of matching each tile to those that it overlaps with
                    n = len(mif_tile_df)
                    all_costs = []
                    all_shifts = []

                    for current_idx in range(n):
                        # TODO: probably is at this level I have to add the shift max lim
                        costs, shifts = match_tiles(mif_tile_df, reference_idx=current_idx, min_overlap_percent = 2)
                        all_costs.append(costs)
                        all_shifts.append(shifts)

                    mif_tile_df["stitching_costs"] = all_costs
                    mif_tile_df["stitching_shifts"] = all_shifts


                    # Step 1: Build the cost matrix which I will use as adjacency for the min span tree
                    adj_matrix = build_adjacency_matrix_from_costs(mif_tile_df, cost_column='stitching_costs')

                    # Step 2: Create sparse matrix and compute MST
                    graph_sparse = csr_matrix(adj_matrix)
                    mst = minimum_spanning_tree(graph_sparse)
                    # The MST will be used to calculate the transofrmation matrices between each tile and a reference tile.
                    # For the moment I just pick 0 as reference but maybe there is a better way, in general I dont think it matters much.                
                    transform_dict = build_transform_dict_from_mst(mif_tile_df, mst, reference_tile=0)
                    
                    # apply transform, user inputs are transform_dict and mif_tile_df, output is the stitched_img
                    stitched_img = apply_transforms_and_stitch(mif_tile_df, transform_dict, reference_tile=0)



                    # Save the full image as a TIFF file
                    tiff.imwrite(output_tif_path, np.flipud(stitched_img))

                    mif_tile_df.to_csv(output_cc_path, index=False)

                    # Convert NumPy arrays to lists for JSON compatibility
                    json_ready_dict = {k: v.tolist() for k, v in transform_dict.items()}

                    # Save to JSON file
                    with open(output_jason_path, "w") as f:
                        json.dump(json_ready_dict, f, indent=2)
                        
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")
            
            



File with '.ve-mif' extension found: MosaicInfo_S_001_2087154489.ve-mif
🔄 Stitching image for S_1...

Processing reference tile 0...

Comparing reference 0 to tile 0...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 1...
Overlap %: 5
overlap img0 x: 0-8192, y 0-410
overlap img1 x: 0-8192, y 7782-8192
Detected pixel offset (row, col): [-72. -90.]

Processing reference tile 1...

Comparing reference 1 to tile 0...
Overlap %: 5
overlap img0 x: 0-8192, y 7782-8192
overlap img1 x: 0-8192, y 0-410
Detected pixel offset (row, col): [72. 90.]

Comparing reference 1 to tile 1...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]
buffer in pixels based on DF output: 250
Total image will be of size including buffer: 8692.0x16474.0

Index: 0, Filename: Z:\Alma\plant4-wafer-1\alma-plant-4_data\session_1084385689\Site 1\S_001_2087154489\Tile_r1-c1_S_001_2087154489.tif
t00
(2,)

Index: 1, Filename: Z:\Alma\plant4-wafer-1\alma-plant-

In [ ]:
from atlas.io import extract_s_number

buffer_in_microns = 40
max_shift_in_pixles = 400

for raw_data_folder in series_list:
    # Check if files in the folder have the ".ve-tie" extension
    tie_file = None
    mif_file = None
    for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
        #if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
        #    print(f"File with '.ve-tie' extension found: {file.name}")
        #    tie_file = file
        if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file

            mif_tile_df = get_tiles_dataframe(mif_file, buffer_microns=buffer_in_microns)

            #total_img_width, total_img_height = get_total_canvas_size(mif_tile_df)
            # check if stitching is already available
            first_tif_path = raw_data_folder.joinpath(Path(mif_tile_df.iloc[0]['Filename']).name)
            extracted_number = extract_s_number(first_tif_path)
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")
                #stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=max_shift_in_pixles)

                try:
                    stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=400)
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")

            #fig, ax = plt.subplots(1, 1, figsize=(15, 15))
            #ax.imshow(stitched_img[:,:], cmap='gray')